In [25]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

In [2]:
data=pd.read_csv('Ecommerce_data.csv')
print(data.head())

                                                Text                   label
0  Urban Ladder Eisner Low Back Study-Office Comp...               Household
1  Contrast living Wooden Decorative Box,Painted ...               Household
2  IO Crest SY-PCI40010 PCI RAID Host Controller ...             Electronics
3  ISAKAA Baby Socks from Just Born to 8 Years- P...  Clothing & Accessories
4  Indira Designer Women's Art Mysore Silk Saree ...  Clothing & Accessories


In [27]:
print(data.iloc[4]["Text"])

Indira Designer Women's Art Mysore Silk Saree With Blouse Piece (Star-Red) This Saree Is Of Art Mysore Silk & Comes With Blouse Piece.


# **Without Preprocess**

## **without using pipeline**

In [16]:
v=TfidfVectorizer()
X_wp=v.fit_transform(data.Text)
print(X_wp.toarray()[0])

[0. 0. 0. ... 0. 0. 0.]


In [35]:
y_wp=data.label
X_train_wp,X_test_wp,y_train_wp,y_test_wp=train_test_split(X_wp,y_wp,test_size=0.2,random_state=42,stratify=y_wp)

In [22]:
model_wp=MultinomialNB()
model_wp.fit(X_train_wp,y_train_wp)
y_pred_wp=model_wp.predict(X_test_wp)
report_wp=classification_report(y_test_wp,y_pred_wp)
print(report_wp)

                        precision    recall  f1-score   support

                 Books       0.98      0.94      0.96      1200
Clothing & Accessories       0.98      0.98      0.98      1200
           Electronics       0.97      0.97      0.97      1200
             Household       0.93      0.97      0.95      1200

              accuracy                           0.96      4800
             macro avg       0.97      0.96      0.97      4800
          weighted avg       0.97      0.96      0.97      4800



## **with Pipeline**

In [36]:
X_p=data.Text
y_p=data.label
X_train_p,X_test_p,y_train_p,y_test_p=train_test_split(X_p,y_p,test_size=0.2,random_state=42,stratify=y_p)

In [24]:
model_p=Pipeline([
    ('tfidf',TfidfVectorizer()),
    ('nb',MultinomialNB())
])

model_p.fit(X_train_p,y_train_p)
y_pred_p=model_p.predict(X_test_p)
report_p=classification_report(y_test_p,y_pred_p)
print(report_p)

                        precision    recall  f1-score   support

                 Books       0.98      0.94      0.96      1200
Clothing & Accessories       0.98      0.98      0.98      1200
           Electronics       0.97      0.97      0.97      1200
             Household       0.93      0.97      0.95      1200

              accuracy                           0.97      4800
             macro avg       0.97      0.97      0.97      4800
          weighted avg       0.97      0.97      0.97      4800



# **With Preprocessing**

In [33]:
nlp=spacy.load("en_core_web_sm")

def preprocess(text):
  doc=nlp(text)
  filterd_tokens=[]
  for token in doc:
    if token.is_stop or token.is_punct:
      continue
    filterd_tokens.append(token.lemma_)
  return " ".join(filterd_tokens)
text=data.iloc[0]["Text"]
print(len(text))
print(len(preprocess(text)))

597
397


In [34]:
data['new_Text']=data.Text.apply(preprocess)
print(data.head())

                                                Text  ...                                           new_Text
0  Urban Ladder Eisner Low Back Study-Office Comp...  ...  Urban Ladder Eisner Low Study Office Computer ...
1  Contrast living Wooden Decorative Box,Painted ...  ...  contrast live Wooden Decorative Box Painted Bo...
2  IO Crest SY-PCI40010 PCI RAID Host Controller ...  ...  IO Crest SY PCI40010 PCI RAID Host Controller ...
3  ISAKAA Baby Socks from Just Born to 8 Years- P...  ...  ISAKAA Baby Socks bear 8 Years- Pack 4 6 8 12 ...
4  Indira Designer Women's Art Mysore Silk Saree ...  ...  Indira Designer woman Art Mysore Silk Saree Bl...

[5 rows x 3 columns]


In [37]:
X=data['new_Text']
y=data['label']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [39]:
model=Pipeline([
    ('tfidf',TfidfVectorizer()),
    ('nb',MultinomialNB())
])
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

                        precision    recall  f1-score   support

                 Books       0.98      0.94      0.96      1200
Clothing & Accessories       0.97      0.98      0.98      1200
           Electronics       0.97      0.97      0.97      1200
             Household       0.94      0.97      0.95      1200

              accuracy                           0.96      4800
             macro avg       0.97      0.96      0.96      4800
          weighted avg       0.97      0.96      0.96      4800

